# **SETUP & LOAD DATA**

In [27]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("IoT Guardian Big Data Analytics") \
    .getOrCreate()

#**DATASET OVERVIEW**

In [28]:
train_df = spark.read.parquet("/content/train.parquet")
val_df = spark.read.parquet("/content/val.parquet")
test_df = spark.read.parquet("/content/test.parquet")

print("Train Rows:", train_df.count())
print("Validation Rows:", val_df.count())
print("Test Rows:", test_df.count())

Train Rows: 132085
Validation Rows: 28304
Test Rows: 28305


In [29]:
def rename_columns(df):
    for c in df.columns:
        df = df.withColumnRenamed(c, c.replace(".", "_"))
    return df

train_df = rename_columns(train_df)
val_df = rename_columns(val_df)
test_df = rename_columns(test_df)

In [30]:
train_df.printSchema()

root
 |-- frame_time_delta: double (nullable = true)
 |-- frame_time_relative: double (nullable = true)
 |-- frame_len: long (nullable = true)
 |-- ip_src: string (nullable = true)
 |-- ip_dst: string (nullable = true)
 |-- tcp_srcport: long (nullable = true)
 |-- tcp_dstport: long (nullable = true)
 |-- tcp_flags: string (nullable = true)
 |-- tcp_time_delta: double (nullable = true)
 |-- tcp_len: long (nullable = true)
 |-- tcp_ack: long (nullable = true)
 |-- tcp_connection_fin: double (nullable = true)
 |-- tcp_connection_rst: double (nullable = true)
 |-- tcp_connection_sack: double (nullable = true)
 |-- tcp_connection_syn: double (nullable = true)
 |-- tcp_flags_ack: long (nullable = true)
 |-- tcp_flags_fin: long (nullable = true)
 |-- tcp_flags_push: long (nullable = true)
 |-- tcp_flags_reset: long (nullable = true)
 |-- tcp_flags_syn: long (nullable = true)
 |-- tcp_flags_urg: long (nullable = true)
 |-- tcp_hdr_len: long (nullable = true)
 |-- tcp_payload: string (nullable 

In [31]:
from pyspark.sql.functions import col

train_df.groupBy("label") \
    .count() \
    .show()

+-----+-----+
|label|count|
+-----+-----+
|    0|75997|
|    1|56088|
+-----+-----+



In [32]:
train_df.groupBy("class") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()

+--------------------+-----+
|               class|count|
+--------------------+-----+
|              Attack|56088|
|   patientMonitoring|53963|
|environmentMonito...|22034|
+--------------------+-----+



In [33]:
print(train_df.columns)

['frame_time_delta', 'frame_time_relative', 'frame_len', 'ip_src', 'ip_dst', 'tcp_srcport', 'tcp_dstport', 'tcp_flags', 'tcp_time_delta', 'tcp_len', 'tcp_ack', 'tcp_connection_fin', 'tcp_connection_rst', 'tcp_connection_sack', 'tcp_connection_syn', 'tcp_flags_ack', 'tcp_flags_fin', 'tcp_flags_push', 'tcp_flags_reset', 'tcp_flags_syn', 'tcp_flags_urg', 'tcp_hdr_len', 'tcp_payload', 'tcp_pdu_size', 'tcp_window_size_value', 'tcp_checksum', 'mqtt_clientid', 'mqtt_clientid_len', 'mqtt_conack_flags', 'mqtt_conack_val', 'mqtt_conflag_passwd', 'mqtt_conflag_qos', 'mqtt_conflag_reserved', 'mqtt_conflag_retain', 'mqtt_conflag_willflag', 'mqtt_conflags', 'mqtt_dupflag', 'mqtt_hdrflags', 'mqtt_kalive', 'mqtt_len', 'mqtt_msg', 'mqtt_msgtype', 'mqtt_qos', 'mqtt_retain', 'mqtt_topic', 'mqtt_topic_len', 'mqtt_ver', 'mqtt_willmsg_len', 'ip_proto', 'ip_ttl', 'class', 'label']


#**DATA QUALITY ASSESSMENT**

In [34]:
from pyspark.sql.functions import col, count, when

missing_values = train_df.select([
    count(when(col(f"`{c}`").isNull(), c)).alias(c.replace(".", "_"))
    for c in train_df.columns
])

missing_values.show()

+----------------+-------------------+---------+------+------+-----------+-----------+---------+--------------+-------+-------+------------------+------------------+-------------------+------------------+-------------+-------------+--------------+---------------+-------------+-------------+-----------+-----------+------------+---------------------+------------+-------------+-----------------+-----------------+---------------+-------------------+----------------+---------------------+-------------------+---------------------+-------------+------------+-------------+-----------+--------+--------+------------+--------+-----------+----------+--------------+--------+----------------+--------+------+-----+-----+
|frame_time_delta|frame_time_relative|frame_len|ip_src|ip_dst|tcp_srcport|tcp_dstport|tcp_flags|tcp_time_delta|tcp_len|tcp_ack|tcp_connection_fin|tcp_connection_rst|tcp_connection_sack|tcp_connection_syn|tcp_flags_ack|tcp_flags_fin|tcp_flags_push|tcp_flags_reset|tcp_flags_syn|tcp_fla

In [35]:
total_rows = train_df.count()
unique_rows = train_df.distinct().count()

print(f"Total Rows      : {total_rows}")
print(f"Unique Rows     : {unique_rows}")
print(f"Duplicate Rows  : {total_rows - unique_rows}")

Total Rows      : 132085
Unique Rows     : 132083
Duplicate Rows  : 2


In [36]:
num_rows = train_df.count()
num_cols = len(train_df.columns)

print(f"Number of Rows    : {num_rows}")
print(f"Number of Columns : {num_cols}")

Number of Rows    : 132085
Number of Columns : 52


In [37]:
dtype_df = spark.createDataFrame(
    train_df.dtypes,
    ["Column", "Data Type"]
)

dtype_df.show(100, truncate=False)

+---------------------+---------+
|Column               |Data Type|
+---------------------+---------+
|frame_time_delta     |double   |
|frame_time_relative  |double   |
|frame_len            |bigint   |
|ip_src               |string   |
|ip_dst               |string   |
|tcp_srcport          |bigint   |
|tcp_dstport          |bigint   |
|tcp_flags            |string   |
|tcp_time_delta       |double   |
|tcp_len              |bigint   |
|tcp_ack              |bigint   |
|tcp_connection_fin   |double   |
|tcp_connection_rst   |double   |
|tcp_connection_sack  |double   |
|tcp_connection_syn   |double   |
|tcp_flags_ack        |bigint   |
|tcp_flags_fin        |bigint   |
|tcp_flags_push       |bigint   |
|tcp_flags_reset      |bigint   |
|tcp_flags_syn        |bigint   |
|tcp_flags_urg        |bigint   |
|tcp_hdr_len          |bigint   |
|tcp_payload          |string   |
|tcp_pdu_size         |double   |
|tcp_window_size_value|bigint   |
|tcp_checksum         |string   |
|mqtt_clientid

In [38]:
from pyspark.sql.functions import *

label_dist = (
    train_df
    .groupBy("label")
    .count()
    .withColumn(
        "percentage",
        round(
            col("count") * 100 / train_df.count(),
            2
        )
    )
)

label_dist.show()

+-----+-----+----------+
|label|count|percentage|
+-----+-----+----------+
|    0|75997|     57.54|
|    1|56088|     42.46|
+-----+-----+----------+



In [39]:
attack_class_dist = (
    train_df
    .groupBy("class")
    .count()
    .orderBy(desc("count"))
)

attack_class_dist.show(50, truncate=False)

+---------------------+-----+
|class                |count|
+---------------------+-----+
|Attack               |56088|
|patientMonitoring    |53963|
|environmentMonitoring|22034|
+---------------------+-----+



In [40]:
clean_df = train_df

for c in train_df.columns:
    clean_df = clean_df.withColumnRenamed(
        c,
        c.replace(".", "_")
    )

clean_df.select(
    "frame_time_delta",
    "frame_len",
    "tcp_window_size_value",
    "mqtt_len",
    "ip_ttl"
).describe().show()

+-------+-------------------+------------------+---------------------+------------------+------------------+
|summary|   frame_time_delta|         frame_len|tcp_window_size_value|          mqtt_len|            ip_ttl|
+-------+-------------------+------------------+---------------------+------------------+------------------+
|  count|             132085|            132085|               132085|            132085|            132085|
|   mean|0.07722248811751614|165.59089222848922|   2340.8482189499186|27.940674565620622| 72.35534693568536|
| stddev| 0.4024242946638586| 330.5947705576487|    7075.597250953538| 55.93216352197969|21.562325974441595|
|    min|                0.0|                54|                    0|               0.0|                64|
|    max|          44.436313|              1766|                65535|             692.0|               128|
+-------+-------------------+------------------+---------------------+------------------+------------------+



In [43]:
from pyspark.sql.functions import col, round

label_summary = (
    train_df
    .groupBy("label")
    .count()
    .withColumn(
        "percentage",
        round(col("count") * 100 / train_df.count(), 2)
    )
)

label_summary.show()

+-----+-----+----------+
|label|count|percentage|
+-----+-----+----------+
|    0|75997|     57.54|
|    1|56088|     42.46|
+-----+-----+----------+



#**DEVICE RISK ANALYTICS**

**4.1 Device Traffic Summary**

In [45]:
from pyspark.sql.functions import *

device_summary = (
    train_df
    .groupBy("ip_src")
    .agg(
        count("*").alias("total_packets"),
        sum("label").alias("attack_packets"),
        avg("frame_time_delta").alias("avg_time_delta"),
        avg("tcp_window_size_value").alias("avg_window_size")
    )
)

device_summary.show(20, truncate=False)

+------------+-------------+--------------+---------------------+-----------------+
|ip_src      |total_packets|attack_packets|avg_time_delta       |avg_window_size  |
+------------+-------------+--------------+---------------------+-----------------+
|10.5.150.156|990          |0             |6.85030303030302E-5  |512.0            |
|10.5.126.155|599          |0             |1.2132220367278796E-4|512.0            |
|10.5.126.162|185          |0             |7.095135135135137E-5 |512.0            |
|10.5.126.163|195          |0             |1.6370769230769237E-4|512.0            |
|10.5.126.131|601          |0             |5.059234608985018E-5 |512.0            |
|10.5.126.161|192          |0             |0.015937713541666662 |512.0            |
|10.5.126.136|952          |0             |9.371008403361342E-5 |512.0            |
|10.5.126.144|291          |0             |8.602749140893471E-5 |512.0            |
|10.5.126.168|4793         |0             |1.3707135405800078E-4|512.0      

**4.2 Device Risk Score**

In [51]:
from pyspark.sql.functions import *

max_packets = device_summary.agg(
    max("total_packets")
).collect()[0][0]

device_risk = (
    device_summary
    .withColumn(
        "attack_rate",
        col("attack_packets") / col("total_packets")
    )
    .withColumn(
        "traffic_weight",
        col("total_packets") / lit(max_packets)
    )
    .withColumn(
        "risk_score",
        round(
            (
                0.8 * col("attack_rate")
                +
                0.2 * col("traffic_weight")
            ) * 100,
            2
        )
    )
)

**4.3 Risk Level Classification**

In [56]:
from pyspark.sql.functions import *

device_risk = (
    device_risk
    .withColumn(
        "risk_level",
        when(col("risk_score") >= 80, "Critical")
        .when(col("risk_score") >= 50, "High")
        .when(col("risk_score") >= 20, "Medium")
        .otherwise("Low")
    )
)

device_risk.orderBy(
    desc("risk_score")
).show(20, truncate=False)

+------------+-------------+--------------+---------------------+-----------------+-----------+--------------------+----------+----------+
|ip_src      |total_packets|attack_packets|avg_time_delta       |avg_window_size  |attack_rate|traffic_weight      |risk_score|risk_level|
+------------+-------------+--------------+---------------------+-----------------+-----------+--------------------+----------+----------+
|192.168.1.90|32396        |32396         |0.007593037288553789 |5186.545777256451|1.0        |1.0                 |100.0     |Critical  |
|192.168.1.91|17244        |17244         |0.003956356123868798 |3492.269948967757|1.0        |0.5322879367823188  |90.65     |Critical  |
|10.16.120.44|3579         |3579          |1.2280217937971416E-4|5943.871751886002|1.0        |0.11047660204963576 |82.21     |Critical  |
|10.16.120.72|2869         |2869          |2.5127570582084312E-5|7237.126524921576|1.0        |0.08856031608840598 |81.77     |Critical  |
|10.5.126.147|7487         

**4.4 Top 10 High-Risk Devices**

In [57]:
top_devices = (
    device_risk
    .orderBy(desc("risk_score"))
    .limit(10)
)

top_devices.show(truncate=False)

+------------+-------------+--------------+---------------------+-----------------+-----------+-------------------+----------+----------+
|ip_src      |total_packets|attack_packets|avg_time_delta       |avg_window_size  |attack_rate|traffic_weight     |risk_score|risk_level|
+------------+-------------+--------------+---------------------+-----------------+-----------+-------------------+----------+----------+
|192.168.1.90|32396        |32396         |0.007593037288553789 |5186.545777256451|1.0        |1.0                |100.0     |Critical  |
|192.168.1.91|17244        |17244         |0.003956356123868798 |3492.269948967757|1.0        |0.5322879367823188 |90.65     |Critical  |
|10.16.120.44|3579         |3579          |1.2280217937971416E-4|5943.871751886002|1.0        |0.11047660204963576|82.21     |Critical  |
|10.16.120.72|2869         |2869          |2.5127570582084312E-5|7237.126524921576|1.0        |0.08856031608840598|81.77     |Critical  |
|10.5.126.147|7487         |0     

**4.5 Risk Level Distribution**

In [58]:
risk_distribution = (
    device_risk
    .groupBy("risk_level")
    .count()
    .orderBy(desc("count"))
)

risk_distribution.show()

+----------+-----+
|risk_level|count|
+----------+-----+
|       Low|   40|
|  Critical|    4|
+----------+-----+



In [59]:
device_risk.orderBy(desc("risk_score")).show(20, truncate=False)

+------------+-------------+--------------+---------------------+-----------------+-----------+--------------------+----------+----------+
|ip_src      |total_packets|attack_packets|avg_time_delta       |avg_window_size  |attack_rate|traffic_weight      |risk_score|risk_level|
+------------+-------------+--------------+---------------------+-----------------+-----------+--------------------+----------+----------+
|192.168.1.90|32396        |32396         |0.007593037288553789 |5186.545777256451|1.0        |1.0                 |100.0     |Critical  |
|192.168.1.91|17244        |17244         |0.003956356123868798 |3492.269948967757|1.0        |0.5322879367823188  |90.65     |Critical  |
|10.16.120.44|3579         |3579          |1.2280217937971416E-4|5943.871751886002|1.0        |0.11047660204963576 |82.21     |Critical  |
|10.16.120.72|2869         |2869          |2.5127570582084312E-5|7237.126524921576|1.0        |0.08856031608840598 |81.77     |Critical  |
|10.5.126.147|7487         

#**PROTOCOL ANALYTICS**

**5.1 MQTT Topic Analysis**

In [61]:
from pyspark.sql.functions import *

topic_stats = (
    train_df
    .groupBy("mqtt_topic")
    .agg(
        count("*").alias("traffic_count"),
        sum("label").alias("attack_count")
    )
    .withColumn(
        "attack_rate",
        round(
            col("attack_count") * 100 /
            col("traffic_count"),
            2
        )
    )
    .orderBy(desc("traffic_count"))
)

topic_stats.show(20, truncate=False)

+-------------+-------------+------------+-----------+
|mqtt_topic   |traffic_count|attack_count|attack_rate|
+-------------+-------------+------------+-----------+
|0            |44341        |31010       |69.94      |
|EMG          |11939        |0           |0.0        |
|ECG          |11883        |0           |0.0        |
|AirFlow      |11839        |0           |0.0        |
|Pulsoximeter |11795        |0           |0.0        |
|0.0          |6448         |6448        |100.0      |
|CoGas        |2755         |0           |0.0        |
|Smoke        |2747         |0           |0.0        |
|Fire         |2728         |0           |0.0        |
|Temperature  |1775         |0           |0.0        |
|Barometer    |1663         |0           |0.0        |
|SolarRad     |1658         |0           |0.0        |
|Humidity     |1617         |0           |0.0        |
|Topic1       |1008         |1008        |100.0      |
|Glucometer   |93           |0           |0.0        |
|InfusionP

**5.2 MQTT Message Type Analysis**

In [62]:
msgtype_stats = (
    train_df
    .groupBy("mqtt_msgtype")
    .agg(
        count("*").alias("traffic_count"),
        sum("label").alias("attack_count")
    )
    .withColumn(
        "attack_rate",
        round(
            col("attack_count") * 100 /
            col("traffic_count"),
            2
        )
    )
    .orderBy(desc("traffic_count"))
)

msgtype_stats.show(truncate=False)

+------------+-------------+------------+-----------+
|mqtt_msgtype|traffic_count|attack_count|attack_rate|
+------------+-------------+------------+-----------+
|3.0         |80813        |18568       |22.98      |
|0.0         |21695        |21695       |100.0      |
|4.0         |18450        |13020       |70.57      |
|13.0        |4079         |0           |0.0        |
|12.0        |4072         |0           |0.0        |
|2.0         |1424         |1336        |93.82      |
|1.0         |1406         |1323        |94.1       |
|8.0         |62           |62          |100.0      |
|9.0         |54           |54          |100.0      |
|5.0         |28           |28          |100.0      |
|14.0        |2            |2           |100.0      |
+------------+-------------+------------+-----------+



**5.3 Top Risky MQTT Topics**

In [63]:
topic_stats.orderBy(
    desc("attack_rate"),
    desc("traffic_count")
).show(20, truncate=False)

+-----------------------------------------------------------------+-------------+------------+-----------+
|mqtt_topic                                                       |traffic_count|attack_count|attack_rate|
+-----------------------------------------------------------------+-------------+------------+-----------+
|0.0                                                              |6448         |6448        |100.0      |
|Topic1                                                           |1008         |1008        |100.0      |
|///////                                                          |33           |33          |100.0      |
|$SYS/#                                                           |33           |33          |100.0      |
|/../../../../                                                    |32           |32          |100.0      |
|$SYS/broker/version                                              |31           |31          |100.0      |
|#                                   

In [64]:
topic_stats.show(20, truncate=False)

+-------------+-------------+------------+-----------+
|mqtt_topic   |traffic_count|attack_count|attack_rate|
+-------------+-------------+------------+-----------+
|0            |44341        |31010       |69.94      |
|EMG          |11939        |0           |0.0        |
|ECG          |11883        |0           |0.0        |
|AirFlow      |11839        |0           |0.0        |
|Pulsoximeter |11795        |0           |0.0        |
|0.0          |6448         |6448        |100.0      |
|CoGas        |2755         |0           |0.0        |
|Smoke        |2747         |0           |0.0        |
|Fire         |2728         |0           |0.0        |
|Temperature  |1775         |0           |0.0        |
|Barometer    |1663         |0           |0.0        |
|SolarRad     |1658         |0           |0.0        |
|Humidity     |1617         |0           |0.0        |
|Topic1       |1008         |1008        |100.0      |
|Glucometer   |93           |0           |0.0        |
|InfusionP

In [65]:
msgtype_stats.show(truncate=False)

+------------+-------------+------------+-----------+
|mqtt_msgtype|traffic_count|attack_count|attack_rate|
+------------+-------------+------------+-----------+
|3.0         |80813        |18568       |22.98      |
|0.0         |21695        |21695       |100.0      |
|4.0         |18450        |13020       |70.57      |
|13.0        |4079         |0           |0.0        |
|12.0        |4072         |0           |0.0        |
|2.0         |1424         |1336        |93.82      |
|1.0         |1406         |1323        |94.1       |
|8.0         |62           |62          |100.0      |
|9.0         |54           |54          |100.0      |
|5.0         |28           |28          |100.0      |
|14.0        |2            |2           |100.0      |
+------------+-------------+------------+-----------+



**5.4 MQTT Threat Intelligence Panel**

In [66]:
high_risk_topics = (
    topic_stats
    .filter(col("attack_rate") > 50)
    .orderBy(desc("attack_rate"))
)

high_risk_topics.show(truncate=False)

+------------------------------------------------------------------+-------------+------------+-----------+
|mqtt_topic                                                        |traffic_count|attack_count|attack_rate|
+------------------------------------------------------------------+-------------+------------+-----------+
|mqtt-malaria/beem.loadr-MBP-di-Ivan-5.lan-54761-14/data/3616/10000|1            |1           |100.0      |
|mqtt-malaria/beem.loadr-MBP-di-Ivan-5.lan-54761-8/data/6232/10000 |1            |1           |100.0      |
|mqtt-malaria/beem.loadr-MBP-di-Ivan-5.lan-54761-13/data/2489/10000|1            |1           |100.0      |
|mqtt-malaria/beem.loadr-MBP-di-Ivan-5.lan-54761-9/data/4040/10000 |1            |1           |100.0      |
|mqtt-malaria/beem.loadr-MBP-di-Ivan-5.lan-54761-3/data/902/10000  |1            |1           |100.0      |
|mqtt-malaria/beem.loadr-MBP-di-Ivan-5.lan-54761-4/data/1453/10000 |1            |1           |100.0      |
|mqtt-malaria/beem.loadr-MBP

**5.5 Top Risky MQTT Message Types**

In [67]:
high_risk_msgtypes = (
    msgtype_stats
    .filter(col("attack_rate") > 50)
    .orderBy(desc("attack_rate"))
)

high_risk_msgtypes.show(truncate=False)

+------------+-------------+------------+-----------+
|mqtt_msgtype|traffic_count|attack_count|attack_rate|
+------------+-------------+------------+-----------+
|8.0         |62           |62          |100.0      |
|0.0         |21695        |21695       |100.0      |
|14.0        |2            |2           |100.0      |
|5.0         |28           |28          |100.0      |
|9.0         |54           |54          |100.0      |
|1.0         |1406         |1323        |94.1       |
|2.0         |1424         |1336        |93.82      |
|4.0         |18450        |13020       |70.57      |
+------------+-------------+------------+-----------+



#**TRAFFIC BURST & TEMPORAL ANALYTICS**

**6.1 Create Time-Ordered Dataset**

In [68]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

time_df = train_df.select(
    "frame_time_relative",
    "label"
).orderBy("frame_time_relative")

time_df.show(10)

+-------------------+-----+
|frame_time_relative|label|
+-------------------+-----+
|                0.0|    0|
|                0.0|    0|
|                0.0|    1|
|                0.0|    0|
|                0.0|    0|
|                0.0|    1|
|             4.8E-5|    0|
|             5.3E-5|    0|
|             6.0E-5|    1|
|             7.5E-5|    1|
+-------------------+-----+
only showing top 10 rows


**6.2 Rolling Attack Count**

In [69]:
window_spec = Window.orderBy("frame_time_relative")

burst_df = (
    time_df
    .withColumn(
        "rolling_attack_count",
        sum("label").over(
            window_spec.rowsBetween(-100, 0)
        )
    )
)

burst_df.show(20)

+-------------------+-----+--------------------+
|frame_time_relative|label|rolling_attack_count|
+-------------------+-----+--------------------+
|                0.0|    0|                   0|
|                0.0|    1|                   1|
|                0.0|    0|                   1|
|                0.0|    0|                   1|
|                0.0|    0|                   1|
|                0.0|    1|                   2|
|             4.8E-5|    0|                   2|
|             5.3E-5|    0|                   2|
|             6.0E-5|    1|                   3|
|             7.5E-5|    1|                   4|
|             8.9E-5|    1|                   5|
|            1.06E-4|    0|                   5|
|            1.16E-4|    0|                   5|
|            1.23E-4|    0|                   5|
|            1.32E-4|    0|                   5|
|            1.34E-4|    0|                   5|
|            1.48E-4|    0|                   5|
|            1.55E-4

**6.3 Identify Peak Attack Windows**

In [70]:
peak_bursts = (
    burst_df
    .orderBy(desc("rolling_attack_count"))
)

peak_bursts.show(20, truncate=False)

+-------------------+-----+--------------------+
|frame_time_relative|label|rolling_attack_count|
+-------------------+-----+--------------------+
|0.01637            |1    |101                 |
|0.0189879999999999 |1    |101                 |
|0.016382           |1    |101                 |
|0.016828           |1    |101                 |
|0.0189389999999999 |1    |101                 |
|0.016978           |1    |101                 |
|0.015209           |1    |101                 |
|0.017074           |1    |101                 |
|0.015236           |1    |101                 |
|0.017289           |1    |101                 |
|0.015403           |1    |101                 |
|0.017296           |1    |101                 |
|0.015416           |1    |101                 |
|0.017302           |1    |101                 |
|0.015651           |1    |101                 |
|0.017309           |1    |101                 |
|0.01636            |1    |101                 |
|0.016797           

**6.4 Average Attack Density**

In [71]:
burst_df.select(
    avg("rolling_attack_count")
).show()

+-------------------------+
|avg(rolling_attack_count)|
+-------------------------+
|        42.88820077980088|
+-------------------------+



**6.5 Attack vs Normal Traffic Over Time**

In [72]:
attack_timeline = (
    train_df
    .groupBy("label")
    .agg(
        count("*").alias("packet_count"),
        avg("frame_time_relative").alias("avg_time_position")
    )
)

attack_timeline.show()

+-----+------------+------------------+
|label|packet_count| avg_time_position|
+-----+------------+------------------+
|    0|       75997|2364.2241112412153|
|    1|       56088| 45.14871962792422|
+-----+------------+------------------+



**6.6 High-Risk Time Periods**

In [73]:
high_risk_periods = (
    burst_df
    .filter(col("rolling_attack_count") > 50)
)

high_risk_periods.show(20)

+-------------------+-----+--------------------+
|frame_time_relative|label|rolling_attack_count|
+-------------------+-----+--------------------+
|           0.004017|    1|                  51|
|           0.004028|    1|                  52|
|            0.00424|    1|                  53|
|           0.004254|    1|                  54|
|           0.004443|    1|                  55|
|           0.004469|    1|                  56|
|           0.004485|    1|                  57|
|            0.00449|    1|                  58|
|           0.004668|    0|                  58|
|           0.004706|    0|                  58|
|           0.004736|    0|                  57|
|            0.00492|    1|                  58|
|           0.004935|    1|                  58|
|           0.004949|    1|                  59|
|           0.004953|    1|                  60|
|           0.004962|    1|                  61|
|           0.005248|    0|                  60|
|           0.005297

**6.7 Traffic Burst Statistics**

In [75]:
burst_df.select(
    min("rolling_attack_count").alias("Min"),
    avg("rolling_attack_count").alias("Average"),
    max("rolling_attack_count").alias("Max")
).show()

+---+-----------------+---+
|Min|          Average|Max|
+---+-----------------+---+
|  0|42.88820077980088|101|
+---+-----------------+---+



**6.8 Burst Severity Categories**

In [76]:
from pyspark.sql.functions import *

burst_levels = (
    burst_df
    .withColumn(
        "burst_level",
        when(col("rolling_attack_count") >= 80, "Extreme")
        .when(col("rolling_attack_count") >= 50, "High")
        .when(col("rolling_attack_count") >= 20, "Medium")
        .otherwise("Low")
    )
)

burst_levels.groupBy(
    "burst_level"
).count().show()

+-----------+-----+
|burst_level|count|
+-----------+-----+
|        Low|71883|
|     Medium| 2780|
|       High| 4057|
|    Extreme|53365|
+-----------+-----+



**6.9 Attack Density Summary**

In [77]:
attack_density = (
    burst_df
    .groupBy("label")
    .agg(
        avg("rolling_attack_count").alias("avg_burst_density"),
        max("rolling_attack_count").alias("max_burst_density")
    )
)

attack_density.show()

+-----+-----------------+-----------------+
|label|avg_burst_density|max_burst_density|
+-----+-----------------+-----------------+
|    0|4.395831414397937|              100|
|    1|95.04382399087149|              101|
+-----+-----------------+-----------------+



#**7.0 SPARK SQL THREAT INTELLIGENCE**

**7.1**

In [78]:
train_df.createOrReplaceTempView("iot_traffic")

**7.2 Attack Distribution**

In [79]:
spark.sql("""
SELECT
    label,
    COUNT(*) AS total_packets
FROM iot_traffic
GROUP BY label
ORDER BY total_packets DESC
""").show()

+-----+-------------+
|label|total_packets|
+-----+-------------+
|    0|        75997|
|    1|        56088|
+-----+-------------+



**7.3 Top Attack Classes**

In [80]:
spark.sql("""
SELECT
    class,
    COUNT(*) AS total_records
FROM iot_traffic
GROUP BY class
ORDER BY total_records DESC
""").show(20, False)

+---------------------+-------------+
|class                |total_records|
+---------------------+-------------+
|Attack               |56088        |
|patientMonitoring    |53963        |
|environmentMonitoring|22034        |
+---------------------+-------------+



**7.4 Top MQTT Topics**

In [81]:
spark.sql("""
SELECT
    mqtt_topic,
    COUNT(*) AS traffic_count,
    SUM(label) AS attack_count
FROM iot_traffic
GROUP BY mqtt_topic
ORDER BY attack_count DESC
""").show(20, False)

+-----------------------------------------------------------------+-------------+------------+
|mqtt_topic                                                       |traffic_count|attack_count|
+-----------------------------------------------------------------+-------------+------------+
|0                                                                |44341        |31010       |
|0.0                                                              |6448         |6448        |
|Topic1                                                           |1008         |1008        |
|///////                                                          |33           |33          |
|$SYS/#                                                           |33           |33          |
|/../../../../                                                    |32           |32          |
|$SYS/broker/version                                              |31           |31          |
|#                                                

#**EXPORT RESULT**

In [82]:
device_risk.write.mode("overwrite").parquet("outputs/device_risk")

topic_stats.write.mode("overwrite").parquet("outputs/topic_stats")

msgtype_stats.write.mode("overwrite").parquet("outputs/msgtype_stats")

In [83]:
device_risk.write.mode("overwrite") \
    .option("header", True) \
    .csv("outputs_csv/device_risk")

topic_stats.write.mode("overwrite") \
    .option("header", True) \
    .csv("outputs_csv/topic_stats")

msgtype_stats.write.mode("overwrite") \
    .option("header", True) \
    .csv("outputs_csv/msgtype_stats")